In [6]:
import pandas as pd
import folium
from folium.plugins import TimestampedGeoJson
from branca.colormap import LinearColormap
import os
import json

# ────────────────────────────────────────────────
# Load data functions (same as before)
# ────────────────────────────────────────────────
def load_data(file_path, coords_csv="HarmfulAlgalBloom_MonitoringSites_-7518154866587238262.csv"):
    if not os.path.exists(file_path):
        print(f"⚠️ Main data file '{file_path}' not found. Using empty dataset.")
        df = pd.DataFrame()
    else:
        if file_path.endswith(('.xlsx', '.xls')):
            df = pd.read_excel(file_path, sheet_name=0)
        else:
            df = pd.read_csv(file_path)
        df['Date_Sample_Collected'] = pd.to_datetime(df['Date_Sample_Collected'], errors='coerce')
        if 'Result_Name' in df.columns:
            df['Result_Name'] = (
                df['Result_Name']
                .astype(str)
                .str.strip()
                .str.replace(r'\s+', ' ', regex=True)
                .str.replace('\xa0', ' ', regex=False)
            )
    
    if not os.path.exists(coords_csv):
        raise FileNotFoundError(f"⚠️ Coordinates file '{coords_csv}' not found.")
    coords_df = pd.read_csv(coords_csv, encoding='utf-8')
    df = df.merge(coords_df, on="Site_Description", how="left")
    df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
    df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
    return df

def load_community(file_path="MASTER spreadsheet of community summaries.xlsx"):
    if not os.path.exists(file_path):
        print(f"⚠️ Community data file '{file_path}' not found. Using empty dataset.")
        return pd.DataFrame()
    
    df = pd.read_excel(file_path, sheet_name=0)
    df.columns = df.columns.str.strip()
    if 'Lat' in df.columns:
        df = df.rename(columns={'Lat': 'Latitude'})
    if 'Long' in df.columns:
        df = df.rename(columns={'Long': 'Longitude'})
    
    if not pd.api.types.is_datetime64_any_dtype(df['Date']):
        df['Date'] = pd.to_datetime(df['Date'], origin='1899-12-30', errors='coerce')
    
    date_idx = df.columns.get_loc('Date')
    total_idx = df.columns.get_loc('Total plankton')
    species_cols = df.columns[date_idx + 1 : total_idx + 1].tolist()
    
    melted_df = pd.melt(df,
                        id_vars=['Location', 'Latitude', 'Longitude', 'Date'],
                        value_vars=species_cols,
                        var_name='Result_Name',
                        value_name='Result_Value_Numeric')
    
    melted_df['Site_Description'] = melted_df['Location']
    melted_df['Date_Sample_Collected'] = melted_df['Date']
    melted_df = melted_df.drop(['Location', 'Date'], axis=1)
    melted_df['Result_Value_Numeric'] *= 1000
    melted_df['Units'] = 'cells/L'
    melted_df['Result_Name'] = (
        melted_df['Result_Name']
        .astype(str)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.replace('\xa0', ' ', regex=False)
    )
    melted_df['Result_Name'] += ' *'
    melted_df['Latitude'] = pd.to_numeric(melted_df['Latitude'], errors='coerce')
    melted_df['Longitude'] = pd.to_numeric(melted_df['Longitude'], errors='coerce')
    return melted_df

# ────────────────────────────────────────────────
# Main script
# ────────────────────────────────────────────────
file_path = "HarmfulAlgalBloom_MonitoringSites_4208500738590205542.csv"
coords_csv = "site_coordinates.csv"
df = load_data(file_path, coords_csv)
community_df = load_community()

# Combine and filter for Karenia
combined_df = pd.concat([df, community_df], ignore_index=True)
karenia_mask = combined_df['Result_Name'].str.contains('Karenia', na=False)
filtered_df = combined_df[karenia_mask & combined_df['Result_Value_Numeric'].notna()].copy()
filtered_df = filtered_df.sort_values('Date_Sample_Collected')

# Colormap (same as before)
colormap = LinearColormap(
    colors=['#641478', '#89CFF0', '#21908c', '#5dc863', '#fde725'],
    index=[0, 100000, 200000, 300000, 500000],
    vmin=0,
    vmax=500000
)

# Build GeoJSON features
features = []
for _, row in filtered_df.iterrows():
    if pd.notna(row['Latitude']) and pd.notna(row['Longitude']):
        value = row['Result_Value_Numeric']
        color = colormap(value)
        feature = {
            'type': 'Feature',
            'geometry': {
                'type': 'Point',
                'coordinates': [row['Longitude'], row['Latitude']]
            },
            'properties': {
                'time': row['Date_Sample_Collected'].isoformat(),
                'style': {
                    'color': color,
                    'fillColor': color,
                    'fillOpacity': 0.8,
                    'radius': 6
                },
                'icon': 'circle',
                'iconstyle': {
                    'fillColor': color,
                    'fillOpacity': 0.8,
                    'stroke': 'true',
                    'radius': 6
                },
                'popup': (
                    f"<b>{row['Site_Description']}</b><br>"
                    f"{row['Date_Sample_Collected'].date()}<br>"
                    f"{row['Result_Name']}<br>"
                    f"{value:,.0f} {row.get('Units', 'cells/L')}"
                )
            }
        }
        features.append(feature)

geojson = {
    'type': 'FeatureCollection',
    'features': features
}

# Create map
m = folium.Map(
    location=[-34.9, 138.6],
    zoom_start=7,
    control_scale=True
)

# TimestampedGeoJson with non-cumulative / sliding window effect
TimestampedGeoJson(
    geojson,
    period='P7D',               # slider steps by 1 week
    duration='P7D',             # each point visible for 14 days → shows roughly last 2 weeks
    # duration='P7D',           # alternative: strict 1-week visibility
    add_last_point=True,         # keep most recent points visible
    auto_play=False,
    loop=False,
    max_speed=0.25,
    loop_button=True,
    date_options='YYYY-MM-DD',
    time_slider_drag_update=True
).add_to(m)

# Tiles (same)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr='Esri', name='Satellite', overlay=False, control=True
).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}",
    attr='Esri', name='Labels', overlay=True, control=True
).add_to(m)
folium.LayerControl().add_to(m)

# Fit bounds
#if not filtered_df.empty:
#    lat_min = filtered_df['Latitude'].min()
#    lon_min = filtered_df['Longitude'].min()
#    lat_max = filtered_df['Latitude'].max()
#    lon_max = filtered_df['Longitude'].max()
#    if pd.notna(lat_min) and pd.notna(lon_min) and pd.notna(lat_max) and pd.notna(lon_max):
#        m.fit_bounds([[lat_min, lon_min], [lat_max, lon_max]])

# Save
m.save('karenia_animation_sliding_window.html')
print("Animation saved to 'karenia_animation_sliding_window.html'. Open in browser.")
print("Try tweaking duration='P7D', 'P14D', 'P30D' etc. to see what gives the best 'moving bloom' effect.")

Animation saved to 'karenia_animation_sliding_window.html'. Open in browser.
Try tweaking duration='P7D', 'P14D', 'P30D' etc. to see what gives the best 'moving bloom' effect.
